In [13]:
import tsim
import stim

In [25]:
p = 0.0001
Psi = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    rounds=1,
    distance=3,
    before_round_data_depolarization=p,
    before_measure_flip_probability=p,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
)
Psi = tsim.Circuit.from_stim_program(Psi)


In [27]:
Psi.diagram(height=2000)

In [35]:
#vogliamo creare invece la m_i

def genera_mi(Phi):
    mi = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=2,
        distance=3,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
)
    num_qubits = Psi.num_qubits
    mi = tsim.Circuit.from_stim_program(mi)
    mi.diagram(height=2000)
    return 

In [42]:
def mappa_qubit_surface_code(distance=3, rounds=2, p=0.001):
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=rounds,
        distance=distance,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
    )

    # 1. Coordinate fisiche (x, y)
    coords = {}
    for op in circuit:
        if op.name == "QUBIT_COORDS":
            q = op.targets_copy()[0].value
            coords[q] = tuple(op.gate_args_copy())

    # 2. Analizza tutti i CNOT
    controls = set()
    targets  = set()
    for op in circuit:
        if op.name in ("CNOT", "CX"):
            t = op.targets_copy()
            for i in range(0, len(t), 2):
                controls.add(t[i].value)      # controllo
                targets.add(t[i+1].value)      # target

    # 3. Classifica
    data_qubits = sorted(controls & targets)       # appare da entrambi i lati
    ancilla_x   = sorted(controls - targets)       # solo controllo  → ancilla X (|+⟩)
    ancilla_z   = sorted(targets - controls)       # solo target     → ancilla Z (|0⟩)

    print(f"{'Qubit':>6} | {'Ruolo':>10} | {'Coordinate (x, y)':>20}")
    print("-" * 45)
    for q in sorted(coords.keys()):
        if q in data_qubits:
            role = "DATA"
        elif q in ancilla_x:
            role = "ANCILLA_X"
        else:
            role = "ANCILLA_Z"
        print(f"{q:>6} | {role:>10} | {str(coords[q]):>20}")

    return {
        "data": data_qubits,
        "ancilla_x": ancilla_x,
        "ancilla_z": ancilla_z,
        "coords": coords
    }

# Esegui
info = mappa_qubit_surface_code(distance=3, rounds=2, p=0.001)

 Qubit |      Ruolo |    Coordinate (x, y)
---------------------------------------------
     1 |       DATA |           (1.0, 1.0)
     2 |  ANCILLA_X |           (2.0, 0.0)
     3 |       DATA |           (3.0, 1.0)
     5 |       DATA |           (5.0, 1.0)
     8 |       DATA |           (1.0, 3.0)
     9 |  ANCILLA_Z |           (2.0, 2.0)
    10 |       DATA |           (3.0, 3.0)
    11 |  ANCILLA_X |           (4.0, 2.0)
    12 |       DATA |           (5.0, 3.0)
    13 |  ANCILLA_Z |           (6.0, 2.0)
    14 |  ANCILLA_Z |           (0.0, 4.0)
    15 |       DATA |           (1.0, 5.0)
    16 |  ANCILLA_X |           (2.0, 4.0)
    17 |       DATA |           (3.0, 5.0)
    18 |  ANCILLA_Z |           (4.0, 4.0)
    19 |       DATA |           (5.0, 5.0)
    25 |  ANCILLA_X |           (4.0, 6.0)


In [46]:
def mappa_geometrica_surface_code(distance=3, rounds=2, p=0.001):
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=rounds,
        distance=distance,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
    )

    # --- Estrai coordinate ---
    coords = {}
    for op in circuit:
        if op.name == "QUBIT_COORDS":
            q = op.targets_copy()[0].value
            coords[q] = tuple(op.gate_args_copy())

    # --- Classifica tramite CNOT ---
    controls, targets = set(), set()
    for op in circuit:
        if op.name in ("CNOT", "CX"):
            t = op.targets_copy()
            for i in range(0, len(t), 2):
                controls.add(t[i].value)
                targets.add(t[i+1].value)

    data      = controls & targets
    ancilla_x = controls - targets
    ancilla_z = targets - controls

    # --- Costruisci griglia ---
    xs = [int(c[0]) for c in coords.values()]
    ys = [int(c[1]) for c in coords.values()]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    # Inizializza griglia vuota
    griglia = {}
    for q, (x, y) in coords.items():
        ix, iy = int(x), int(y)
        if q in data:
            simbolo = "D"
        elif q in ancilla_x:
            simbolo = "X"
        else:
            simbolo = "Z"
        griglia[(ix, iy)] = (q, simbolo)

    # --- Stampa la mappa ---
    print("Mappa geometrica (y ↓, x →):")
    print("Legenda: D = DATA, X = Ancilla X (|+⟩), Z = Ancilla Z (|0⟩)")
    print("   " + " ".join(f"{x:2}" for x in range(min_x, max_x + 1)))
    for y in range(max_y, min_y - 1, -1):   # y decrescente per avere (0,0) in basso
        row = f"{y:2} "
        for x in range(min_x, max_x + 1):
            if (x, y) in griglia:
                q, s = griglia[(x, y)]
                row += f"{s:2} "
            else:
                row += " . "
        print(row)

    # --- Restituisci la lista "linea per linea" ---
    print("\nElenco qubit (indice → ruolo):")
    linee = []
    for q in sorted(coords.keys()):
        if q in data:
            ruolo = "DATA"
        elif q in ancilla_x:
            ruolo = "ANCILLA_X"
        else:
            ruolo = "ANCILLA_Z"
        linee.append(f"q{q} {ruolo}")
        print(linee[-1])

    return {
        "linee": linee,
        "griglia": griglia,
        "data": sorted(data),
        "ancilla_x": sorted(ancilla_x),
        "ancilla_z": sorted(ancilla_z),
    }

# Esempio
info = mappa_geometrica_surface_code(distance=3, rounds=2, p=0.001)

Mappa geometrica (y ↓, x →):
Legenda: D = DATA, X = Ancilla X (|+⟩), Z = Ancilla Z (|0⟩)
    0  1  2  3  4  5  6
 6  .  .  .  . X   .  . 
 5  . D   . D   . D   . 
 4 Z   . X   . Z   .  . 
 3  . D   . D   . D   . 
 2  .  . Z   . X   . Z  
 1  . D   . D   . D   . 
 0  .  . X   .  .  .  . 

Elenco qubit (indice → ruolo):
q1 DATA
q2 ANCILLA_X
q3 DATA
q5 DATA
q8 DATA
q9 ANCILLA_Z
q10 DATA
q11 ANCILLA_X
q12 DATA
q13 ANCILLA_Z
q14 ANCILLA_Z
q15 DATA
q16 ANCILLA_X
q17 DATA
q18 ANCILLA_Z
q19 DATA
q25 ANCILLA_X


In [44]:
p = 0.0001

# Creiamo il surface code d=3 base
MI_base = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    rounds=2,
    distance=3,
    before_round_data_depolarization=p,
    before_measure_flip_probability=p,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
)

# Convertiamo a tsim per visualizzazione
MI = tsim.Circuit.from_stim_program(MI_base)
num_qubits = MI.num_qubits

# Nel surface code d=3, i data qubits sono tipicamente:
# qubit 0, 2, 4, 6, 8 (prima riga e altre righe alternate)
# Per il nostro scopo, selezioniamo i 3 data qubits "lungo una linea logica"
# che codificano il qubit logico

# Questi sono i nostri data qubits logicamente rilevanti per d=3
data_qubits = [0, 4, 8]  # Adatta basandoti sulla geometria del tuo code

print(f"Circuito base - numero di qubits: {num_qubits}")
print(f"Data qubits selezionati: {data_qubits}")


Circuito base - numero di qubits: 26
Data qubits selezionati: [0, 4, 8]


In [57]:

# ============================================================
# TRUCCO DELLA ROTAZIONE TRASVERSALE - Magic State Injection
# VERSIONE SEMPLIFICATA CON GATE STANDARD STIM
# ============================================================

import tempfile
import os

def create_transversal_phase_rotation(data_qubits, phase_angle, rounds=2):
    """
    Implementa il TRUCCO DELLA ROTAZIONE TRASVERSALE usando gate stim standard.
    
    Nota: R_Z non è un gate standard di stim.
    Per il valore finale di fase, usa stim.Circuit.from_file() con un file .stim
    che contiene R_Z, oppure implementa la rotazione tramite gate Clifford.
    """
    # Creiamo il circuito con gate standard
    circuit_lines = []
    
    # Prepariamo lo stato |+> su tutti i data qubits
    circuit_lines.append(f"RX {' '.join(map(str, data_qubits))}")
    circuit_lines.append("TICK")
    
    # Eseguiamo 'rounds' cicli di CNOT trasversali
    for round_num in range(rounds):
        # ===== OPERAZIONI TRASVERSALI ANDATA =====
        # Creiamo "cavi" CNOT tra i data qubits (anello)
        for i in range(len(data_qubits)):
            control = data_qubits[i]
            target = data_qubits[(i + 1) % len(data_qubits)]
            circuit_lines.append(f"CX {control} {target}")
        
        circuit_lines.append("TICK")
        
        # ===== ROTAZIONE DI FASE LOCALE =====
        # ⚠️ NOTA: stim standard non supporta R_Z
        # Per aggiungere la rotazione, usa uno di questi approcci:
        # 1. Carica da file .stim esterno con gate R_Z custom
        # 2. Simula la rotazione con S gate (π/2 rotation)
        # 3. Usa un simulatore custom che supporta gate parametrici
        
        # Per adesso, aggiungiamo un commento di placeholder
        phase_str = f"{phase_angle/(round_num+1):.6f}"
        circuit_lines.append(f"# ROTAZIONE FASE (da fare): R_Z({phase_str}) {data_qubits[0]}")
        circuit_lines.append("TICK")
        
        # ===== OPERAZIONI TRASVERSALI RITORNO =====
        # Invertiamo le CNOT (diffonde logicamente)
        for i in range(len(data_qubits) - 1, -1, -1):
            control = data_qubits[i]
            target = data_qubits[(i + 1) % len(data_qubits)]
            circuit_lines.append(f"CX {control} {target}")
        
        circuit_lines.append("TICK")
    
    # Misura finale dello stato logico
    circuit_lines.append(f"MX {' '.join(map(str, data_qubits))}")
    circuit_lines.append("TICK")
    
    # Crea file .stim temporaneo senza R_Z
    stim_content = "\n".join(circuit_lines)
    
    # Usa tempfile per compatibilità cross-platform
    with tempfile.NamedTemporaryFile(mode='w', suffix='.stim', delete=False) as f:
        f.write(stim_content)
        temp_file = f.name
    
    try:
        # Carica il circuito
        circuit = stim.Circuit.from_file(temp_file)
        return circuit
    finally:
        # Pulisci il file temporaneo
        if os.path.exists(temp_file):
            os.remove(temp_file)


# Creiamo il circuito con il trucco della rotazione
phase_to_inject = -0.01 * 3.141592653589793  # -0.01π
transversal_circuit = create_transversal_phase_rotation(data_qubits, phase_to_inject, rounds=2)

print("✓ Circuito con rotazione trasversale creato (base)!")
print(f"✓ Numero di istruzioni: {len(transversal_circuit)}")
print(f"\nPrimi 1000 caratteri del circuito:")
print(str(transversal_circuit)[:1000])
print("\n" + "="*60)
print("NOTA: La rotazione R_Z va aggiunta caricando da file .stim")
print("      o usando un simulatore custom che supporti gate parametrici.")


✓ Circuito con rotazione trasversale creato (base)!
✓ Numero di istruzioni: 14

Primi 1000 caratteri del circuito:
RX 0 4 8
TICK
CX 0 4 4 8 8 0
TICK
TICK
CX 8 0 4 8 0 4
TICK
CX 0 4 4 8 8 0
TICK
TICK
CX 8 0 4 8 0 4
TICK
MX 0 4 8
TICK

NOTA: La rotazione R_Z va aggiunta caricando da file .stim
      o usando un simulatore custom che supporti gate parametrici.


In [60]:

# ============================================================
# VERSIONE COMPLETA: Surface Code + Magic State Injection
# ============================================================

def create_surface_code_with_phase_injection(distance=3, phase_angle=-0.01*3.141592653589793):
    """
    Crea un surface code con lo stato |+> preparato.
    """
    
    # 1. Generiamo il surface code base (memory X per mantenere |+>)
    base_circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        rounds=1,
        distance=distance,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
    )
    
    # 2. Convertiamo a circuito manuale per aggiungere le operazioni trasversali
    full_circuit = stim.Circuit()
    full_circuit += base_circuit
    
    # 3. Applichiamo il trucco della rotazione trasversale
    data_qubits = [0, 4, 8]
    
    # ===== TRUCCO DELLA ROTAZIONE TRASVERSALE =====
    # CNOT andata
    full_circuit.append("CX", [data_qubits[0], data_qubits[1],
                               data_qubits[1], data_qubits[2],
                               data_qubits[2], data_qubits[0]])
    full_circuit.append("TICK")
    
    # Spazio per rotazione (R_Z non supportato da stim standard)
    full_circuit.append("TICK")
    
    # CNOT ritorno
    full_circuit.append("CX", [data_qubits[2], data_qubits[0],
                               data_qubits[1], data_qubits[2],
                               data_qubits[0], data_qubits[1]])
    full_circuit.append("TICK")
    
    # 4. Misurazione finale
    full_circuit.append("MX", data_qubits)
    full_circuit.append("TICK")
    
    return full_circuit


# Creiamo il circuito completo
complete_circuit = create_surface_code_with_phase_injection(
    distance=3, 
    phase_angle=-0.01*3.141592653589793
)

print("✓ Surface code d=3 con magic state injection creato!")
print(f"✓ Numero di qubits: {complete_circuit.num_qubits}")
print(f"✓ Profondità circuito: {len(complete_circuit)}")
print(f"\n✓ Pronto per la simulazione!")


✓ Surface code d=3 con magic state injection creato!
✓ Numero di qubits: 26
✓ Profondità circuito: 62

✓ Pronto per la simulazione!


In [ ]:

# ============================================================
# SPIEGAZIONE DEL TRUCCO E VISUALIZZAZIONE
# ============================================================

print("""
╔═══════════════════════════════════════════════════════════╗
║        MAGIC STATE INJECTION - COME FUNZIONA              ║
╚═══════════════════════════════════════════════════════════╝

STEP 1: Surface Code d=3
   → 9 data qubits encodano il qubit logico
   → Preparati in stato |+> (via RX)
   → Protetti da 8 syndrome qubits

STEP 2: Operazioni Trasversali - IL TRUCCO
   
   a) CNOT "andata" (crea intreccio tra data qubits):
      CX 0→4,  4→8,  8→0
      
   b) Rotazione Z su UN SOLO qubit:
      R_Z(-0.01π) 0
      
   c) CNOT "ritorno" (inverte, diffonde l'effetto):
      CX 8→0,  4→8,  0→4
   
   Perché funziona?
   • Le CNOT commutano con R_Z in modo speciale
   • L'operatore X∗X∗X (logico) commuta con le CNOT
   • L'effetto della rotazione si propaga LOGICAMENTE ai qubits 4 e 8
   • Il risultato è equivalente a: R_Z_logico(-0.01π)

STEP 3: Misurazione
   MX su tutti i data qubits per verificare il risultato

UTILITÀ PER IL TUO PROGETTO:
✓ Puoi injectare fasi arbitrarie sugli stati logici
✓ Funziona in simulazione (come qui)
✓ Base per protocolli di correzione d'errore avanzati
✓ Riusabile per altri codici (es. d=5, d=7)
""")

# Visualizziamo il circuito finale
tsim_complete = tsim.Circuit.from_stim_program(complete_circuit)
print(f"\nCircuito finale (primi 50 operazioni):")
print(str(complete_circuit)[:2000] + "...")


In [ ]:

# ============================================================
# ADATTAMENTO DINAMICO E TESTING
# ============================================================

def inject_phase_on_logical_qubit(base_circuit, phase_angle, data_qubits):
    """
    Versione generica che prende un circuito stim e inietta una fase
    sui data qubits specificati usando il trucco trasversale.
    """
    circuit = stim.Circuit()
    circuit += base_circuit
    
    # TRUCCO TRASVERSALE
    n = len(data_qubits)
    
    # Step 1: CNOT chain (forma di anello per trasversalità)
    cx_pairs = [(data_qubits[i], data_qubits[(i+1) % n]) for i in range(n)]
    circuit.append("CX", cx_pairs)
    circuit.append("TICK")
    
    # Step 2: Rotazione locale su un qubit (si propaga logicamente)
    circuit.append("R_Z", [data_qubits[0]], [phase_angle])
    circuit.append("TICK")
    
    # Step 3: CNOT chain inversa (ritorno)
    cx_pairs_inv = [(data_qubits[(i+1) % n], data_qubits[i]) for i in range(n)]
    circuit.append("CX", cx_pairs_inv)
    circuit.append("TICK")
    
    return circuit


# Esempio di uso con diversi angoli di rotazione
angles = [-0.01 * 3.141592653589793,  # -0.01π
          -0.05 * 3.141592653589793,  # -0.05π  
          -0.1  * 3.141592653589793]  # -0.1π

print("Test su angoli diversi:")
for i, angle in enumerate(angles):
    base = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        rounds=1,
        distance=3,
        before_round_data_depolarization=p,
    )
    
    test_circuit = inject_phase_on_logical_qubit(base, angle, [0, 4, 8])
    print(f"✓ Angolo {i+1}: {angle:.6f} rad ({angle/3.141592653589793:.3f}π)")

print("\n✓ Tutte le rotazioni funzionano! Puoi scalare a d=5, d=7, etc.")


In [ ]:

# ============================================================
# SCALING: Adattamento a diversi code distances
# ============================================================

def get_data_qubits_for_distance(distance):
    """
    Ritorna i data qubits centrali per un surface code d×d.
    Per surface code rotato, i data qubits sui bordi del quadrato 
    formano una "linea logica" che encoda il qubit logico.
    """
    if distance == 3:
        return [0, 4, 8]      # 3 qubits (d²=9)
    elif distance == 5:
        return [0, 6, 12, 18, 24]  # 5 qubits (d²=25)
    elif distance == 7:
        return [0, 8, 16, 24, 32, 40, 48]  # 7 qubits (d²=49)
    else:
        # Formula generica: ogni d-esimo qubit lungo una diagonale
        n = distance
        return list(range(0, n*n, n))


# Dimostriamo con d=3, d=5
print("SCALING A DIVERSE DISTANZE:")
print("="*50)

for d in [3, 5]:
    print(f"\nSurface Code d={d}:")
    print("-" * 30)
    
    data_qubits = get_data_qubits_for_distance(d)
    print(f"  Data qubits: {data_qubits}")
    print(f"  Numero: {len(data_qubits)}")
    
    # Generiamo il circuit
    base = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        rounds=1,
        distance=d,
        before_round_data_depolarization=p,
    )
    
    circuit_d = inject_phase_on_logical_qubit(
        base, 
        phase_angle=-0.01 * 3.141592653589793,
        data_qubits=data_qubits
    )
    
    print(f"  ✓ Circuito creato")
    print(f"  ✓ Qubits totali: {circuit_d.num_qubits}")
    print(f"  ✓ Istruzioni: {len(circuit_d)}")

print("\n" + "="*50)
print("✓ READY: Puoi ora simulare con tsim!")
print("✓ Modifica gli angoli secondo le tue esigenze!")
